In [1]:
'''
The Agent-as-a-Tool pattern allows one agent to delegate a task to another agent.

This is different from a sub-agent. When Agent A calls Agent B as a tool, Agent B's response is passed back to Agent A.
Agent A then uses that information to form its own final response to the user. 
It's a powerful way to compose complex behaviors from simpler, focused, and reusable agents.
------------------------------------------------------------------------------------------------------------------------------

How It Works

Our top-level agent, the trip_data_concierge_agent, acts as the Orchestrator. It has two tools at its disposal:
call_db_agent: A function that internally calls our db_agent to fetch raw data.
call_concierge_agent: A function that calls the concierge_agent.
The concierge_agent itself has a tool: the food_critic_agent.

The flow for a complex query is:
User asks the trip_data_concierge_agent for a hotel and a nearby restaurant.
The Orchestrator first calls call_db_agent to get hotel data.
The data is saved in tool_context.state.
The Orchestrator then calls call_concierge_agent, which retrieves the hotel data from the context.
The concierge_agent receives the request and decides it needs to use its own tool, the food_critic_agent.
The food_critic_agent provides a witty recommendation.
The concierge_agent gets the critic's response and politely formats it.
This final, polished response is returned to the Orchestrator, which presents it to the user.

'''



"\nThe Agent-as-a-Tool pattern allows one agent to delegate a task to another agent.\n\nThis is different from a sub-agent. When Agent A calls Agent B as a tool, Agent B's response is passed back to Agent A.\nAgent A then uses that information to form its own final response to the user. \nIt's a powerful way to compose complex behaviors from simpler, focused, and reusable agents.\n------------------------------------------------------------------------------------------------------------------------------\n\nHow It Works\n\nOur top-level agent, the trip_data_concierge_agent, acts as the Orchestrator. It has two tools at its disposal:\ncall_db_agent: A function that internally calls our db_agent to fetch raw data.\ncall_concierge_agent: A function that calls the concierge_agent.\nThe concierge_agent itself has a tool: the food_critic_agent.\n\nThe flow for a complex query is:\nUser asks the trip_data_concierge_agent for a hotel and a nearby restaurant.\nThe Orchestrator first calls call

In [18]:
import os
import sys
import  json
import asyncio
import random
import string
from uuid import uuid4
from typing import List,Any
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, Markdown, display

#----------ADK , Agent and Evaluation components Tools Contextimports here------------------

from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content , Part
from dotenv import load_dotenv

import  asyncio
from google.adk.tools import ToolContext
from google.adk.tools.agent_tool  import AgentTool
# Assume 'db_agent' is a pre-defined NL2SQL Agent
# For this example, we'll create placeholder agents

print(" All libraries are imported!")

 All libraries are imported!


In [19]:
load_dotenv()

True

In [20]:
#Runner to Help run the agent: This is a HELPER function
async def run_agent_query(agent:Agent,query:str,session : Session,user_id: str,is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n Running query for agent: '{agent.name}' in session: '{session.id}'...")
    runner = Runner(
        agent = agent,
        session_service = session_service,
        app_name = agent.name)
    final_response = ""
    try:
        async for event in runner.run_async(user_id = user_id ,session_id = session.id,new_message = Content(parts=[Part(text = query)],role ="user")):
            if not is_router:
                # Let's see what the agent is thinking! through events 
                print(f"EVENT:{event}")
                if event.is_final_response():
                    final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
        print("\n" + "-"*50)
        print("✅ Final Response:")
        display(Markdown(final_response))
        print("-"*50 + "\n")
    return final_response

In [21]:
# --- Initializing Session Service ---
session_service = InMemorySessionService()
my_user_id = "adk_user_001"

In [22]:
db_agent = Agent(
    name = "db_agent",
    model = "gemini-3.5-flash",
    instruction = "You are a database agent. When asked for data, return this mock JSON object: {'status': 'success', 'data': [{'name': 'The Grand Hotel', 'rating': 5, 'reviews': 450}, {'name': 'Seaside Inn', 'rating': 4, 'reviews': 620}]}"
)

In [23]:
# --------------------- 1. Specialist Agents -----------------------------

# The Food Critic remains the deepest specialist
food_critic_agent = Agent(
    name = "food_critic_agent",
    model = "gemini-3.5-flash",
    instruction="You are a snobby but brilliant food critic. You ONLY respond with a single, witty restaurant suggestion near the provided location.",
)

In [24]:
# --- 2. Tools for the Orchestrator ---
async def call_db_agent(question : str,tool_context : ToolContext,):
    """
    Use this tool FIRST to connect to the database and retrieve a list of places, like hotels or landmarks.
    """
    print("--- TOOL CALL: call_db_agent ---")
    agent_tool = AgentTool(agent = db_agent)
    db_agent_output = await agent_tool.run_async(
        args={"request":question},tool_context = tool_context
    )
    # Storing the retrieved data in the context's state
    tool_context.state["retrieved_data"] = db_agent_output
    return db_agent_output


async def call_concierge_agent(question : str, tool_context : ToolContext):
    """
    After getting data with call_db_agent, use this tool to get travel advice, opinions, or recommendations.
    """
    print("--- TOOL CALL: call_concierge_agent ---")
    # Retrieving the data fetched by the previous tool
    input_data = tool_context.state.get("retrieved_data","No data found.")
    # Formulating a new prompt for the concierge, giving it the data context
    question_with_data = f""" 
    Context: The database returned the following data: {input_data}
    User's Request: {question}
    """
    agent_tool = AgentTool(agent = concierge_agent)
    concierge_output = await  agent_tool.run_async(
        args = {"request": question_with_data},tool_context  = tool_context
    )
    return concierge_output
    

In [25]:
# -----------------------------3.Top-Level Orchestrator Agent -------------------------------
trip_data_concierge_agent = Agent(
    name = "trip_data_concierge",
    model="gemini-3.5-flash",
    description = "Top-level agent that queries a database for travel data, then calls a concierge agent for recommendations.",
    tools = [call_db_agent, call_concierge_agent],
    instruction = """
    You are a master travel planner who uses data to make recommendations.
    1.  **ALWAYS start with the `call_db_agent` tool** to fetch a list of places (like hotels) that match the user's criteria.
    2.  After you have the data, **use the `call_concierge_agent` tool** to answer any follow-up questions for recommendations, opinions, or advice related to the data you just found.
    """,
)
print(f"Orchestrator Agent '{trip_data_concierge_agent.name}' is defined and ready.")

Orchestrator Agent 'trip_data_concierge' is defined and ready.


In [26]:
# ----------------------Testing the Trip Data Concierge Agent --------------------
async def run_trip_data_concierge():
    """
    Sets up a session and runs a query against the top-level
    trip_data_concierge_agent.
    """
    # Creating a new, single-use session for this query
    concierge_session = await session_service.create_session(app_name=trip_data_concierge_agent.name,user_id = my_user_id)
    # This query is specifically designed to trigger the full two-step process:
    # 1. Get data from the db_agent.
    # 2. Get a recommendation from the concierge_agent based on that data.
    query = "Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews."
    print(f"User Query: '{query}'")

    # We call our existing helper function with the top-level orchestrator agent
    await run_agent_query(trip_data_concierge_agent, query, concierge_session, my_user_id)

# Run the test
await run_trip_data_concierge()

User Query: 'Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews.'

 Running query for agent: 'trip_data_concierge' in session: 'b501963a-d755-4109-9200-1d1744f82bd0'...


C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\google\adk\models\llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


EVENT:model_version='gemini-3.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'question': 'Find top-rated hotels in San Francisco, including their rating and number of reviews.'
        },
        id='call_868538',
        name='call_db_agent'
      ),
      thought_signature=b"\x12\xff\x05\n\xfc\x05\x01\x11M2\x0f-Z\x98+\xb7\xc0\x17\xfaZ\xbf\x94O\xf1x[R\xc8\xe2'P!:\xbf\xb7\x0e\x1dO55`\xc8\xd3\xae\x99_\x04\xa8\xe5Q\x8f\x96=\xeb\xb3\x15pS\xae\x1c\t\xeeZ\xc7\xd5\xa1\x14\xdb\xef)\x97l\xdcMB\xa1!d\x8b(\xb2l\x07\xfc\x87\r\xcd\r\xfb\x99\x16\xdb)\xdd\xc4\xe0\x88...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None turn_complete_reason=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=34,
  prompt_token_count=330,
  prompt_tokens_details=[
    ModalityTokenC

Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.


EVENT:model_version=None content=Content(
  parts=[
    Part(
      function_response=FunctionResponse(
        id='call_868538',
        name='call_db_agent',
        response={
          'result': "{'status': 'success', 'data': [{'name': 'The Grand Hotel', 'rating': 5, 'reviews': 450}, {'name': 'Seaside Inn', 'rating': 4, 'reviews': 620}]}"
        }
      )
    ),
  ],
  role='user'
) grounding_metadata=None partial=None turn_complete=None turn_complete_reason=None finish_reason=None error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=None live_session_resumption_update=None live_session_id=None go_away=None voice_activity=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None interaction_id=None environment_id=None invocation_id='e-09b030b7-cb68-4808-bb91-f9f55c3ee9f1' author='trip_data_concierge' actions=EventActions(skip_summarization=None, state_delta={'retrieve

Node execution failed with exception
Traceback (most recent call last):
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\google\adk\workflow\_node_runner.py", line 136, in run
    await self._execute_node(ctx, node_input)
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\google\adk\workflow\_node_runner.py", line 274, in _execute_node
    await self._run_node_loop(ctx, node_input)
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\google\adk\workflow\_node_runner.py", line 288, in _run_node_loop
    async for event in agen:
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\google\adk\workflow\_base_node.py", line 166, in run
    async for item in agen:
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\google\adk\agents\llm_agent.py", line 614, in _run_impl
    async for event in agen:
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\google\adk\workflow\_llm_age

EVENT:model_version=None content=None grounding_metadata=None partial=None turn_complete=None turn_complete_reason=None finish_reason=None error_code='NameError' error_message="name 'concierge_agent' is not defined" interrupted=None custom_metadata=None usage_metadata=None live_session_resumption_update=None live_session_id=None go_away=None voice_activity=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None interaction_id=None environment_id=None invocation_id='e-09b030b7-cb68-4808-bb91-f9f55c3ee9f1' author='trip_data_concierge' actions=EventActions(skip_summarization=None, state_delta={}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=None, end_of_agent=None, agent_state=None, rewind_before_invocation_id=None, route=None, render_ui_widgets=None, set_model_response=None) output=None node_info=NodeInfo(path='trip_data_con

An error occurred: 'NoneType' object has no attribute 'parts'

--------------------------------------------------



Root node trip_data_concierge was cancelled.
Failed to detach context
Traceback (most recent call last):
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\opentelemetry\trace\__init__.py", line 608, in use_span
    yield span
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\opentelemetry\trace\__init__.py", line 508, in start_as_current_span
    yield span
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\opentelemetry\trace\__init__.py", line 443, in start_as_current_span
    yield span
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\google\adk\telemetry\_instrumentation.py", line 78, in record_invocation
    yield
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\google\adk\runners.py", line 699, in _run_node_async
    yield event
GeneratorExit

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\Lenovo\AppDat